In [0]:
%pylab inline

In [0]:
import dataiku
from dataiku import pandasutils as pdu
import pandas as pd

In [0]:
# --- Cell 1 (fixed): download the Kuzu DB to the kernel, then open + dump schema ---
#     Run on a kernel whose code env has `kuzu` (e.g. plugin_visual-graph_managed),
#     so the kuzu version matches the one that built db.kz.
import kuzu, dataiku, os, tempfile, shutil

fol    = dataiku.Folder("5Fbx2exi")
remote = "/built-graphs/8KjiSQ/db.kz"
local  = os.path.join(tempfile.gettempdir(), "primekg_db.kz")

with fol.get_download_stream(remote) as s, open(local, "wb") as f:
    shutil.copyfileobj(s, f)
print("downloaded", os.path.getsize(local) // (1024*1024), "MB ->", local)

conn = kuzu.Connection(kuzu.Database(local, read_only=True))

def q(cypher, **params):
    return conn.execute(cypher, parameters=params or None).get_as_df()

print(q("CALL show_tables() RETURN *"))            # node table + rel table names
print(q("MATCH (a)-[e]->(b) RETURN e LIMIT 1"))    # is `relation` a property?

In [0]:
# --- Cell 2: dwpc_GGD WITH leave-one-out, for one disease ---
# metapath:  g --protein_protein--> m --disease_protein--> D
# weight:    (deg_ppi(g) * deg_ppi(m) * deg_assoc(m) * module_size(D)) ^ -0.4
# LOO:       m != g  AND  g dropped from D's module-size denominator
# Edges are reverse-all'd, so a DIRECTED-out match recovers the true (undirected)
# degree and counts each metapath instance once -> matches the pandas prototype.
#
# ASSUMES: one node table `NODE`, one edge table with a `relation` property.
# If cell 1 shows per-relation rel tables instead, swap the `WHERE e.relation=...`
# filters for typed matches: -[:protein_protein]-> and -[:disease_protein]->.

CYPHER = """
MATCH (g:protein)-[:protein_protein]-(m:protein)-[:disease_protein]-(D:disease)
WHERE D.node_index = $D
  AND m.node_index <> g.node_index
WITH g, m, D,
     CAST(COUNT { MATCH (g)-[:protein_protein]-() } AS DOUBLE) AS ppi_g,
     CAST(COUNT { MATCH (m)-[:protein_protein]-() } AS DOUBLE) AS ppi_m,
     CAST(COUNT { MATCH (m)-[:disease_protein]-() } AS DOUBLE) AS assoc_m,
     CAST(COUNT { MATCH (D)-[:disease_protein]-() } AS DOUBLE) AS mod_raw,
     COUNT { MATCH (g)-[:disease_protein]-(D) } AS g_in_mod
WITH g.node_index AS gene_index, g.node_name AS gene,
     ppi_g, ppi_m, assoc_m, (mod_raw - g_in_mod) AS mod_D
WITH gene_index, gene,
     sum( pow(ppi_g * ppi_m * assoc_m * mod_D, -0.4) ) AS dwpc_GGD
RETURN gene_index, gene, dwpc_GGD
ORDER BY dwpc_GGD DESC
LIMIT 25
"""

print(q(CYPHER, D=12041))   # 12041 = breast cancer (MONDO 7254)

In [0]:
DLIST_Q = """
MATCH (D:disease)-[:disease_protein]-(p:protein)
WITH D.node_index AS disease_index, count(DISTINCT p.node_index) AS module_size
WHERE module_size >= 20
RETURN disease_index, module_size
ORDER BY module_size DESC
"""
dq = q(DLIST_Q)
print("qualifying diseases:", dq.shape[0], "| module sizes:", dq.module_size.min(), "-", dq.module_size.max())
dlist = dq.disease_index.tolist()
dq.head()

In [0]:
CYPHER_ALL = """
MATCH (g:protein)-[:protein_protein]-(m:protein)-[:disease_protein]-(D:disease)
WHERE D.node_index IN $dlist
  AND m.node_index <> g.node_index
WITH g, m, D,
     CAST(COUNT { MATCH (g)-[:protein_protein]-() } AS DOUBLE) AS ppi_g,
     CAST(COUNT { MATCH (m)-[:protein_protein]-() } AS DOUBLE) AS ppi_m,
     CAST(COUNT { MATCH (m)-[:disease_protein]-() } AS DOUBLE) AS assoc_m,
     CAST(COUNT { MATCH (D)-[:disease_protein]-() } AS DOUBLE) AS mod_raw,
     COUNT { MATCH (g)-[:disease_protein]-(D) } AS g_in_mod
WITH g.node_index AS gene_index, D.node_index AS disease_index,
     g_in_mod, ppi_g, ppi_m, assoc_m, (mod_raw - g_in_mod) AS mod_D
WITH gene_index, disease_index,
     max(g_in_mod) AS g_in_mod,
     sum( pow(ppi_g * ppi_m * assoc_m * mod_D, -0.4) ) AS dwpc_GGD
RETURN gene_index, disease_index, dwpc_GGD,
       CASE WHEN g_in_mod > 0 THEN 1 ELSE 0 END AS label
"""

sample = dlist[:20] + [12041]                         # 20 largest modules (incl. breast cancer/obesity); widen later
df = q(CYPHER_ALL, dlist=sample)
print("rows (candidate pairs):", df.shape, "| positives:", int(df.label.sum()))
df.head()

In [0]:
bc = df[df.disease_index == 12041].nlargest(25, "dwpc_GGD").reset_index(drop=True)
print(bc[["gene_index", "dwpc_GGD", "label"]])
# top rows should match Cell-2 values exactly (e.g. SEMA3E 21600 ≈ 0.036407)

In [0]:
print("A pathway_protein edges exist:",
  q("MATCH (a:protein)-[:pathway_protein]-(b:pathway) RETURN count(*) AS c").iloc[0,0])

print("B genes sharing a pathway (double pathway_protein hop):",
  q("""MATCH (g:protein)-[:pathway_protein]-(P:pathway)-[:pathway_protein]-(m:protein)
       WHERE g.node_index <> m.node_index RETURN count(*) AS c""").iloc[0,0])

print("C full GPGD metapath, ANY disease:",
  q("""MATCH (g:protein)-[:pathway_protein]-(P:pathway)-[:pathway_protein]-(m:protein)-[:disease_protein]-(D:disease)
       WHERE g.node_index <> m.node_index RETURN count(*) AS c""").iloc[0,0])

print("D GPGD for breast cancer only (bound disease):",
  q("""MATCH (g:protein)-[:pathway_protein]-(P:pathway)-[:pathway_protein]-(m:protein)-[:disease_protein]-(D:disease)
       WHERE D.node_index = 12041 AND g.node_index <> m.node_index RETURN count(*) AS c""").iloc[0,0])

In [0]:
for rel in ["protein_protein","disease_protein","pathway_protein",
            "pathway_pathway","drug_protein","indication","disease_disease"]:
    c = q(f"MATCH ()-[e:{rel}]-() RETURN count(*) AS c").iloc[0,0]
    print(f"{rel:18s} {c}")

# what does the pathway_protein table actually connect, and do pathway nodes exist?
print(q("CALL show_connection('pathway_protein') RETURN *"))
print("pathway nodes:", q("MATCH (p:pathway) RETURN count(*) AS c").iloc[0,0])

In [0]:


node = dataiku.Dataset("graph_nodes")
node_df = node.get_dataframe()

In [0]:
# Example: load a DSS dataset as a Pandas dataframe
ggd = dataiku.Dataset("dwpc_GGD_sampled")
ggd_df = ggd.get_dataframe()

gcd = dataiku.Dataset("dwpc_gcd_sampled")
gcd_df = gcd.get_dataframe()

gpgd = dataiku.Dataset("dwpc_gpgd_sampled")
gpgd_df = gpgd.get_dataframe()



In [0]:

ga = dataiku.Dataset("guilt_by_association")
ga_df = ga.get_dataframe()


In [0]:
labeled_df = ggd_df.merge(node_df, left_on='gene_index', right_on='node_index')
labeled_df[labeled_df['disease_index']==12041].sort_values(by='dwpc_GGD', ascending=False).head(20)

In [0]:
labeled_df = gcd_df.merge(node_df, left_on='gene_index', right_on='node_index')
labeled_df[labeled_df['disease_index']==12041].sort_values(by='dwpc_GCD', ascending=False).head(20)

In [0]:
labeled_df = gpgd_df.merge(node_df, left_on='gene_index', right_on='node_index')
labeled_df[labeled_df['disease_index']==12041].sort_values(by='dwpc_GPGD', ascending=False).head(20)

In [0]:
labeled_df = ga_df.merge(node_df, left_on='gene_index', right_on='node_index')
labeled_df[labeled_df['disease_index']==12041].sort_values(by='ppi_common_neighbors', ascending=False).head(20)

In [0]:

p_count = dataiku.Dataset("node_centrality")
p_count_df = p_count.get_dataframe()

In [0]:
p_count_df

In [0]:
node_df

In [0]:
labeled_df = p_count_df.merge(node_df, left_on='node_id', right_on="node_index")
labeled_df.sort_values(by='pagerank', ascending=False).head(20)

In [0]:


egdes = dataiku.Dataset("graph_edges")
egdes_df = egdes.get_dataframe()

In [0]:
egdes_df.groupby(['relation', 'display_relation']).count()

In [0]:
egdes_df.groupby('display_relation').count()

In [0]:

va = dataiku.Dataset("validation_set_scored")
va_df = va.get_dataframe()
labeled_df = va_df.merge(node_df, left_on='gene_index', right_on="node_index")
labeled_df[labeled_df['disease_index']==33311].sort_values(by='proba_1', ascending=False).head(20)

In [0]:
labeled_df = va_df.merge(node_df, left_on='gene_index', right_on="node_index")
labeled_df[labeled_df['disease_index']==13108].sort_values(by='proba_1', ascending=False).head(20)

In [0]:

va = dataiku.Dataset("enriched_graph_features_candidate_3")
va_df = va.get_dataframe()
df1 = va_df.groupby("is_target").agg(lambda x: x.isna().astype('float').mean()).transpose()
df1['delta'] = df1.iloc[:, 0] - df1.iloc[:, 1]
df1.sort_values(by='delta', ascending=False)

In [0]:

va2 = dataiku.Dataset("graph_features_candidate_1")
va_df2 = va2.get_dataframe()
df2 = va_df2.groupby("is_target").agg(lambda x: x.isna().astype('float').mean()).transpose()
df2['delta'] = df2.iloc[:, 0] - df2.iloc[:, 1]
df2.sort_values(by='delta', ascending=False)

In [0]:

gd = dataiku.Dataset("gene_disease_edges")
gd_df = gd.get_dataframe()
gd_df['y_id'] = gd_df['y_id'].astype('str')
labeled_df = gd_df.merge(node_df, left_on=['y_id', 'y_type'], right_on=["node_id", "node_type"])
labeled_df[labeled_df['x_id']==2740]

In [0]:
node_df

In [0]:
gd_df